# 01 · Data inventory and ID mapping

Inventory of the downloaded GDC data and mapping of RNA-Seq files to patients. The unit of analysis is the patient, so every file must be traceable to a patient and each patient must contribute one RNA observation.

**Inputs:** `data/raw/`, `data/metadata/`  
**Outputs (`data/processed/`):** `patient_rna_selection.csv`, `rna_selection_audit.csv`, `flow_rna_selection.csv`


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

def find_project_root():
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "notebooks").is_dir() and (folder / "data" / "raw" / "RNA").is_dir():
            return folder.resolve()
    raise FileNotFoundError("Project root not found (needs notebooks/ and data/raw/RNA/).")

ROOT = find_project_root()
RAW, PROC, FIG = ROOT / "data" / "raw", ROOT / "data" / "processed", ROOT / "figures"
PROC.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

def find_folder(prefix):
    hits = [p for p in RAW.iterdir() if p.is_dir() and p.name.lower().startswith(prefix.lower())]
    if len(hits) != 1:
        raise RuntimeError(f"Expected exactly one folder starting with {prefix!r}; found {[p.name for p in hits]}")
    return hits[0]

RNA_DIR = find_folder("RNA")
CLINICAL_DIR = find_folder("clinical.project-tcga-brca")
BIOSPECIMEN_DIR = find_folder("biospecimen.project-tcga-brca")
METADATA_DIR = ROOT / "data" / "metadata"

def read_gdc(path):
    """Read a GDC TSV as text; the GDC missing marker "'--" and empty cells become NaN."""
    df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, low_memory=False)
    return df.replace({"'--": np.nan, "": np.nan})

print("Project root:", ROOT)

import re

SAMPLE_SHEET = METADATA_DIR / "gdc_sample_sheet.2026-10-21.tsv"      # downloaded from the GDC cart with the files
if not SAMPLE_SHEET.is_file():
    raise FileNotFoundError(SAMPLE_SHEET)

Project root: D:\University\Research\tcga-brca-statistics


## 1. What is on disk

Source: NCI Genomic Data Commons (GDC), project TCGA-BRCA, open-access files only, downloaded 2026-10-21. RNA-Seq files are the *STAR - Counts* gene quantification (GENCODE v36).

In [2]:
rna_paths = sorted(RNA_DIR.rglob("*.rna_seq.augmented_star_gene_counts.tsv"))
partial = sorted(RNA_DIR.rglob("*.partial"))
size = np.array([p.stat().st_size for p in rna_paths])
print(f"RNA-Seq files: {len(rna_paths)} | total {size.sum() / 1024**3:.2f} GiB | "
      f"zero-byte: {(size == 0).sum()} | unfinished (.partial): {len(partial)}")
print(f"File size (MiB): min {size.min() / 2**20:.2f}, median {np.median(size) / 2**20:.2f}, max {size.max() / 2**20:.2f}")

RNA-Seq files: 1111 | total 4.39 GiB | zero-byte: 0 | unfinished (.partial): 0
File size (MiB): min 4.01, median 4.05, max 4.07


## 2. Data types and level of each file

Two data types are used: clinical (clinical, follow-up and pathology tables) and RNA-Seq gene expression. The biospecimen tables link samples to patients. One row means different things in different files, so a patient-level table is built before any analysis.


In [3]:
LEVEL = {
    "clinical.tsv": "patient x diagnosis x treatment (nested, repeated)",
    "follow_up.tsv": "patient x follow-up x molecular test (nested, repeated)",
    "pathology_detail.tsv": "diagnosis (one row per patient)",
    "exposure.tsv": "patient exposure record",
    "family_history.tsv": "patient family-history record",
    "sample.tsv": "biospecimen sample",
    "portion.tsv": "portion of a sample",
    "analyte.tsv": "analyte (DNA/RNA extract)",
    "aliquot.tsv": "aliquot",
    "slide.tsv": "tissue slide",
}

def table_summary(path):
    df = read_gdc(path)
    per_patient = df["cases.submitter_id"].value_counts() if "cases.submitter_id" in df else pd.Series(dtype=int)
    return {"table": path.name, "one row is": LEVEL.get(path.name, "-"), "rows": len(df), "columns": df.shape[1],
            "non-empty columns": int(df.notna().any().sum()), "patients": df["cases.submitter_id"].nunique() if "cases.submitter_id" in df else 0,
            "max rows / patient": int(per_patient.max()) if len(per_patient) else 0}

inventory = pd.DataFrame([table_summary(p) for folder in (CLINICAL_DIR, BIOSPECIMEN_DIR) for p in sorted(folder.glob("*.tsv"))])
display(inventory)

,table,one row is,rows,columns,non-empty columns,patients,max rows / patient
0,clinical.tsv,"patient x diagnosis x treatment (nested, repea...",5546,202,74,1098,40
1,exposure.tsv,patient exposure record,1,35,4,1,1
2,family_history.tsv,patient family-history record,0,13,0,0,0
3,follow_up.tsv,"patient x follow-up x molecular test (nested, ...",9427,151,31,1096,25
4,pathology_detail.tsv,diagnosis (one row per patient),1096,83,11,1096,1
5,aliquot.tsv,aliquot,16474,26,25,1098,35
6,analyte.tsv,analyte (DNA/RNA extract),6487,23,18,1098,14
7,portion.tsv,portion of a sample,5626,12,12,1098,13
8,sample.tsv,biospecimen sample,3397,33,15,1098,5
9,slide.tsv,tissue slide,3186,31,19,1098,9


## 3. RNA files to samples and patients

The RNA files contain no patient ID: the **folder name is the GDC file ID**. The GDC sample sheet (downloaded with the files) links each file ID to a sample and a patient. Files are matched by file ID and checked against the file name.

In [4]:
sheet = pd.read_csv(SAMPLE_SHEET, sep="\t", dtype=str, keep_default_na=False)
sheet.columns = [re.sub(r"[^a-z0-9]+", "_", c.strip().lower()).strip("_") for c in sheet.columns]
missing_cols = {"file_id", "file_name", "case_id", "sample_id"} - set(sheet.columns)
assert not missing_cols, f"Sample sheet lacks columns: {missing_cols}"

sheet = sheet.rename(columns={"case_id": "patient_submitter_id", "sample_id": "sample_submitter_id"})
sheet["file_id"] = sheet["file_id"].str.strip().str.lower()
sheet["patient_submitter_id"] = sheet["patient_submitter_id"].str.strip().str.upper()
sheet["sample_submitter_id"] = sheet["sample_submitter_id"].str.strip().str.upper()

disk = pd.DataFrame({
    "file_id": [p.parent.name.strip().lower() for p in rna_paths],           # GDC folder name = file ID
    "disk_filename": [p.name for p in rna_paths],
    "rna_relpath": [p.relative_to(RNA_DIR).as_posix() for p in rna_paths],
})
mapping = sheet.merge(disk, on="file_id", how="outer", indicator=True, validate="one_to_one")
print(mapping["_merge"].value_counts().to_string())
assert (mapping["_merge"] == "both").all(), "Some files are on disk but not in the sample sheet (or the reverse)"
assert (mapping["file_name"].str.casefold() == mapping["disk_filename"].str.casefold()).all(), "File-name mismatch"
mapping = mapping.drop(columns="_merge")

mapping["sample_type_code"] = mapping["sample_submitter_id"].str[13:15]          # TCGA barcode: 01 = primary solid tumour
print("\nSample type code in the TCGA barcode (01 = primary solid tumour):")
print(mapping["sample_type_code"].value_counts().to_string())
for col in ("tissue_type", "tumor_descriptor"):
    if col in mapping:
        print(f"\n{col}:"); print(mapping[col].value_counts().to_string())

print(f"\nFiles: {len(mapping)} | samples: {mapping['sample_submitter_id'].nunique()} | patients: {mapping['patient_submitter_id'].nunique()}")
display(mapping.groupby("patient_submitter_id")["file_id"].nunique().value_counts().sort_index().rename_axis("files per patient").to_frame("patients"))
display(mapping.groupby("patient_submitter_id")["sample_submitter_id"].nunique().value_counts().sort_index().rename_axis("samples per patient").to_frame("patients"))

_merge
both          1111
left_only        0
right_only       0

Sample type code in the TCGA barcode (01 = primary solid tumour):
sample_type_code
01    1111

tissue_type:
tissue_type
Tumor    1111

tumor_descriptor:
tumor_descriptor
Primary    1111

Files: 1111 | samples: 1106 | patients: 1095


,patients
files per patient,
1,1084
2,6
3,5


,patients
samples per patient,
1,1084
2,11


In [5]:
clinical_ids = set(pd.read_csv(CLINICAL_DIR / "clinical.tsv", sep="\t", usecols=["cases.submitter_id"], dtype=str)
                   ["cases.submitter_id"].str.strip().str.upper().dropna())
rna_ids = set(mapping["patient_submitter_id"])
print("Patients in clinical.tsv:", len(clinical_ids))
print("Patients with RNA-Seq:   ", len(rna_ids))
print("In both:                 ", len(clinical_ids & rna_ids))
print("Clinical only:           ", sorted(clinical_ids - rna_ids))
print("RNA only:                ", sorted(rna_ids - clinical_ids))

Patients in clinical.tsv: 1098
Patients with RNA-Seq:    1095
In both:                  1095
Clinical only:            ['TCGA-AC-A5EI', 'TCGA-AR-A0U1', 'TCGA-C8-A9FZ']
RNA only:                 []


## 4. One RNA file per patient

Some patients have several RNA files (several samples, or several aliquots of one sample). To keep one observation per patient, one file is chosen by a fixed rule:

1. smallest sample ID (alphabetical, e.g. `01A` before `01B`);
2. if that sample has several files, smallest aliquot ID;
3. if still tied, smallest file ID.

Aliquot IDs are read from the GDC API only for samples with several files and cached in `data/processed/gdc_aliquot_lookup.csv`.


In [6]:
dup_sample = mapping.duplicated(["patient_submitter_id", "sample_submitter_id"], keep=False)
need_ids = sorted(mapping.loc[dup_sample, "file_id"])
CACHE = PROC / "gdc_aliquot_lookup.csv"

def fetch_aliquots(file_ids):
    import json, requests
    filters = {"op": "in", "content": {"field": "files.file_id", "value": list(file_ids)}}
    r = requests.post("https://api.gdc.cancer.gov/files", timeout=90, data={
        "filters": json.dumps(filters), "format": "JSON", "size": str(len(file_ids)),
        "fields": "file_id,associated_entities.entity_type,associated_entities.entity_submitter_id"})
    r.raise_for_status()
    rows = []
    for hit in r.json()["data"]["hits"]:
        ents = hit.get("associated_entities", [])
        ents = [e for e in ents if e.get("entity_type") == "aliquot"] or ents
        rows.append({"file_id": hit["file_id"].lower(), "aliquot_submitter_id": ents[0]["entity_submitter_id"] if ents else None})
    return pd.DataFrame(rows)

lookup = pd.read_csv(CACHE, dtype=str) if CACHE.exists() else None
if lookup is None or not set(need_ids) <= set(lookup["file_id"]):
    print(f"Querying the GDC API for {len(need_ids)} files ...")
    lookup = fetch_aliquots(need_ids)
    lookup.to_csv(CACHE, index=False)
print("Aliquot IDs available for", lookup["aliquot_submitter_id"].notna().sum(), "of", len(need_ids), "files that need them")

cand = mapping.merge(lookup, on="file_id", how="left", validate="one_to_one")
dup_sample = cand.duplicated(["patient_submitter_id", "sample_submitter_id"], keep=False)
assert cand.loc[dup_sample, "aliquot_submitter_id"].notna().all(), "Aliquot ID missing for a sample with several files"

cand = cand.sort_values(["patient_submitter_id", "sample_submitter_id", "aliquot_submitter_id", "file_id"],
                        na_position="last", kind="stable")
selected = cand.drop_duplicates("patient_submitter_id", keep="first")
assert len(selected) == cand["patient_submitter_id"].nunique() == selected["file_id"].nunique()

cand["selected"] = cand["file_id"].isin(set(selected["file_id"]))
chosen_sample = cand.loc[cand["selected"]].set_index("patient_submitter_id")["sample_submitter_id"]
cand["reason_not_selected"] = np.where(cand["selected"], "",
    np.where(cand["sample_submitter_id"] == cand["patient_submitter_id"].map(chosen_sample),
             "second aliquot of the selected sample", "another sample of the same patient"))

multi = cand.groupby("patient_submitter_id")["file_id"].transform("nunique") > 1
display(cand.loc[multi, ["patient_submitter_id", "sample_submitter_id", "aliquot_submitter_id", "file_id", "selected", "reason_not_selected"]]
        .sort_values(["patient_submitter_id", "sample_submitter_id", "aliquot_submitter_id"]).reset_index(drop=True))

Aliquot IDs available for 10 of 10 files that need them


,patient_submitter_id,sample_submitter_id,aliquot_submitter_id,file_id,selected,reason_not_selected
0,TCGA-A7-A0DB,TCGA-A7-A0DB-01A,TCGA-A7-A0DB-01A-11R-A00Z-07,a8898537-0afa-4655-b123-78c11b90c53a,True,
1,TCGA-A7-A0DB,TCGA-A7-A0DB-01A,TCGA-A7-A0DB-01A-11R-A277-07,6efd37a6-1669-4cdd-95b6-60c688e1e236,False,second aliquot of the selected sample
2,TCGA-A7-A0DB,TCGA-A7-A0DB-01C,NaN,c89590e8-8b74-4c05-a022-e21eab5cffdd,False,another sample of the same patient
3,TCGA-A7-A0DC,TCGA-A7-A0DC-01A,NaN,0e1999e0-9c02-4804-88b6-60b46d100739,True,
4,TCGA-A7-A0DC,TCGA-A7-A0DC-01B,NaN,3e8cff80-ed17-479c-8327-ca8e71ee7482,False,another sample of the same patient
5,TCGA-A7-A13D,TCGA-A7-A13D-01A,TCGA-A7-A13D-01A-13R-A12P-07,50c7c034-fccb-4eb1-b8db-e0a5d9e50172,True,
6,TCGA-A7-A13D,TCGA-A7-A13D-01A,TCGA-A7-A13D-01A-13R-A277-07,25352bc8-c94c-4542-a7da-d4423adf72cf,False,second aliquot of the selected sample
7,TCGA-A7-A13D,TCGA-A7-A13D-01B,NaN,bf6e2b60-d65b-4963-8e8e-49ab3f93c3e1,False,another sample of the same patient
8,TCGA-A7-A13E,TCGA-A7-A13E-01A,TCGA-A7-A13E-01A-11R-A12P-07,0a511373-8fd3-433a-a5e1-877530d6a239,True,
9,TCGA-A7-A13E,TCGA-A7-A13E-01A,TCGA-A7-A13E-01A-11R-A277-07,4b96ee43-874c-4fa6-b5ce-08caf37ef2e1,False,second aliquot of the selected sample


## 5. Flow so far and saved tables

In [7]:
n_not = int((~cand["selected"]).sum())
reasons = cand.loc[~cand["selected"], "reason_not_selected"].value_counts()
flow = pd.DataFrame([
    ("RNA-Seq files on disk (matched 1:1 to the sample sheet)", len(cand), ""),
    ("unique samples", cand["sample_submitter_id"].nunique(), ""),
    ("unique patients", cand["patient_submitter_id"].nunique(), "analysis unit"),
    ("files selected (one per patient)", int(cand["selected"].sum()), "rule in section 4"),
    ("files not selected", n_not, "; ".join(f"{k}: {v}" for k, v in reasons.items())),
], columns=["step", "count", "note"])
display(flow)

keep = ["patient_submitter_id", "sample_submitter_id", "aliquot_submitter_id", "file_id", "file_name", "rna_relpath"]
selected_out = cand.loc[cand["selected"], keep].sort_values("patient_submitter_id").reset_index(drop=True)
selected_out.to_csv(PROC / "patient_rna_selection.csv", index=False)
cand[keep + ["selected", "reason_not_selected"]].to_csv(PROC / "rna_selection_audit.csv", index=False)
flow.to_csv(PROC / "flow_rna_selection.csv", index=False)
print("Saved patient_rna_selection.csv:", selected_out.shape, "| rna_selection_audit.csv:", cand.shape)

,step,count,note
0,RNA-Seq files on disk (matched 1:1 to the samp...,1111,
1,unique samples,1106,
2,unique patients,1095,analysis unit
3,files selected (one per patient),1095,rule in section 4
4,files not selected,16,another sample of the same patient: 11; second...


Saved patient_rna_selection.csv: (1095, 6) | rna_selection_audit.csv: (1111, 17)


**Result.** Every RNA file maps to a sample and a patient. One file per patient gives 1,095 patients; the 16 unused files remain listed in `rna_selection_audit.csv`.